In [1]:
%%writefile main.py
import os
import sys
import random
from collections import defaultdict
import json

from cg.api import (
  AreaType,
  CardType,
  Log,
  LogType,
  Observation,
  SelectContext,
  SelectType,
  OptionType,
  Card,
  Pokemon,
  State,
  all_card_data,
  to_observation_class
)


def read_deck_csv() -> list[int]:
    file_path = "deck.csv"
    if not os.path.exists(file_path):
        file_path = "/kaggle_simulations/agent/" + file_path
    with open(file_path, "r") as file:
        csv = file.read().split("\n")
    return [int(csv[i]) for i in range(60)]

def choose_first(options, option_type):
    for i, op in enumerate(options):
        if op.type == option_type:
            return [i]
    return None

def safe_first_action(obs):
  n = len(obs.select.option)
  min_c = obs.select.minCount
  max_c = obs.select.maxCount

  if max_c == 0:
    return []

  if n == 0:
    raise RuntimeError("No options, but selection requested")
  
  k = min_c
  
  if k == 0 and max_c > 0:
    k = 1
  
  k = min(k, max_c, n)
  
  return list(range(k))



def agent(obs_dict: dict) -> list[int]:
    obs: Observation = to_observation_class(obs_dict)

    if obs.select is None:
        return read_deck_csv()

    options = obs.select.option

    
    if obs.select.context == SelectContext.MAIN:
        for option_type in [
            OptionType.ABILITY,
            OptionType.PLAY,
            OptionType.ATTACH,
            OptionType.ATTACK,
            OptionType.RETREAT,
            OptionType.END,
        ]:
            res = choose_first(options, option_type)
            if res is not None:
                return res
    
    if obs.select.context == SelectContext.ATTACK:
        attack_indices = [
            i for i, op in enumerate(options)
            if op.type == OptionType.ATTACK
        ]
        if attack_indices:
            return [attack_indices[-1]]
        
        
    if obs.select.type == SelectType.YES_NO:
        res = choose_first(options, OptionType.YES)
        if res is not None:
            return res

    return safe_first_action(obs)

Writing main.py


In [2]:
import glob
import os
import tarfile

with tarfile.open("submission.tar.gz", "w:gz") as tar:
    tar.add("main.py", arcname="main.py")
    tar.add(glob.glob('/kaggle/input/**/cg-lib/cg', recursive=True)[0], arcname="cg")
    tar.add(glob.glob('/kaggle/input/**/deck_raging_bolt_v0.csv', recursive=True)[0], arcname="deck.csv")

os.remove('main.py')